In [29]:
import pandas as pd

pipeline = pd.read_csv('../data/sales_pipeline.csv')
teams    = pd.read_csv('../data/sales_teams.csv')
accounts = pd.read_csv('../data/accounts.csv')
products = pd.read_csv('../data/products.csv')

for name, df in [('pipeline',pipeline),('teams',teams),('accounts',accounts),('products',products)]:
    print(name, df.shape)

print('\nFUNNEL:\n', pipeline['deal_stage'].value_counts())
print('\nMISSING VALUES:\n', pipeline.isna().sum())

pipeline (8800, 8)
teams (35, 3)
accounts (85, 7)
products (7, 3)

FUNNEL:
 deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

MISSING VALUES:
 opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64


In [30]:
# parse dates
pipeline['engage_date'] = pd.to_datetime(pipeline['engage_date'])
pipeline['close_date']  = pd.to_datetime(pipeline['close_date'])

# where are the account gaps? (investigate before deciding what to do)
print("Account nulls by stage:\n", pipeline[pipeline['account'].isna()]['deal_stage'].value_counts())

# derived fields
pipeline['is_won']    = pipeline['deal_stage'] == 'Won'
pipeline['is_closed'] = pipeline['deal_stage'].isin(['Won','Lost'])
pipeline['cycle_days'] = (pipeline['close_date'] - pipeline['engage_date']).dt.days

# master table — left joins so every opportunity survives
df = (pipeline
      .merge(teams,    on='sales_agent', how='left')
      .merge(products, on='product',     how='left')
      .merge(accounts, on='account',     how='left'))

print("\nMaster table:", df.shape)

closed = df[df['is_closed']]
print(f"Win rate: {closed['is_won'].mean():.1%}")
print(f"Won revenue: ${df.loc[df['is_won'],'close_value'].sum():,.0f}")

Account nulls by stage:
 deal_stage
Engaging       1088
Prospecting     337
Name: count, dtype: int64

Master table: (8800, 21)
Win rate: 63.2%
Won revenue: $10,005,534


In [31]:
closed = df[df['is_closed']].copy()

# 1) WIN RATE BY SALES AGENT — consistency across the team
agent = (closed.groupby('sales_agent')
         .agg(deals=('is_won','size'), win_rate=('is_won','mean'),
              won_rev=('close_value', lambda s: s[closed.loc[s.index,'is_won']].sum()))
         .sort_values('win_rate'))
print("WIN RATE BY AGENT (worst 5):\n", agent.head())
print("\nWIN RATE BY AGENT (best 5):\n", agent.tail())
print(f"\nSpread: {agent['win_rate'].min():.1%} to {agent['win_rate'].max():.1%}")

# 2) LOST REVENUE — what did losing cost, by product?
lost = closed[~closed['is_won']]
lost_by_product = (lost.groupby('product')
                   .agg(lost_deals=('opportunity_id','size'),
                        avg_price=('sales_price','mean'))
                   .assign(est_lost_rev=lambda x: x['lost_deals']*x['avg_price'])
                   .sort_values('est_lost_rev', ascending=False))
print("\nEST. LOST REVENUE BY PRODUCT:\n", lost_by_product)

# 3) DISCOUNTING — are won deals closing below list price?
won = closed[closed['is_won']].copy()
won['discount'] = 1 - (won['close_value'] / won['sales_price'])
print(f"\nAvg discount on won deals: {won['discount'].mean():.1%}")
print("Discount by product:\n", won.groupby('product')['discount'].mean().sort_values(ascending=False))

WIN RATE BY AGENT (worst 5):
                   deals  win_rate   won_rev
sales_agent                                
Lajuana Vencill     231  0.549784  194632.0
Markita Hansen      227  0.572687  328792.0
Donn Cantrell       275  0.574545  445860.0
Gladys Colclough    232  0.581897  345674.0
Niesha Huffines     175  0.600000  176961.0

WIN RATE BY AGENT (best 5):
                    deals  win_rate   won_rev
sales_agent                                 
Versie Hillebrand    264  0.666667  187693.0
Cecily Lampkin       160  0.668750  229800.0
Wilburn Farren        79  0.696203  157640.0
Maureen Marcano      213  0.699531  350395.0
Hayden Neloms        152  0.703947  272111.0

Spread: 55.0% to 70.4%

EST. LOST REVENUE BY PRODUCT:
                 lost_deals  avg_price  est_lost_rev
product                                            
MG Advanced            430     3393.0     1458990.0
GTX Plus Pro           266     5482.0     1458212.0
GTX Plus Basic         398     1096.0      436208.0
G

In [32]:
# why is discount ~0 and GTXPro NaN?
print("Products in pipeline:\n", sorted(df['product'].unique()))
print("\nProducts in products table:\n", sorted(products['product'].unique()))

# spot-check a few won deals vs list price
print("\n", won[['product','close_value','sales_price']].head(10))

Products in pipeline:
 ['GTK 500', 'GTX Basic', 'GTX Plus Basic', 'GTX Plus Pro', 'GTXPro', 'MG Advanced', 'MG Special']

Products in products table:
 ['GTK 500', 'GTX Basic', 'GTX Plus Basic', 'GTX Plus Pro', 'GTX Pro', 'MG Advanced', 'MG Special']

            product  close_value  sales_price
0   GTX Plus Basic       1054.0       1096.0
1           GTXPro       4514.0          NaN
2       MG Special         50.0         55.0
3        GTX Basic        588.0        550.0
4        GTX Basic        517.0        550.0
5       MG Special         49.0         55.0
6       MG Special         57.0         55.0
7        GTX Basic        601.0        550.0
8   GTX Plus Basic       1026.0       1096.0
10      MG Special         53.0         55.0


In [33]:
# fix the naming mismatch so joins are clean everywhere
df['product'] = df['product'].replace({'GTXPro': 'GTX Pro'})
# (optional) re-merge product info now that names match
df = df.drop(columns=['series','sales_price']).merge(products, on='product', how='left')
print("Unmatched products after fix:", df['sales_price'].isna().sum())  # expect 0

Unmatched products after fix: 0


In [34]:
closed = df[df['is_closed']].copy()

median_wr = closed.groupby('sales_agent')['is_won'].mean().median()
print(f"Median agent win rate: {median_wr:.1%}")

agent_stats = closed.groupby('sales_agent').agg(
    deals=('is_won','size'), wins=('is_won','sum'), win_rate=('is_won','mean'))
below = agent_stats[agent_stats['win_rate'] < median_wr].copy()

below['extra_wins'] = (below['deals'] * median_wr - below['wins']).clip(lower=0)
avg_won_value = closed.loc[closed['is_won'],'close_value'].mean()

extra_wins = below['extra_wins'].sum()
upside = extra_wins * avg_won_value
won_total = closed.loc[closed['is_won'],'close_value'].sum()

print(f"Avg value per won deal: ${avg_won_value:,.0f}")
print(f"Extra wins if below-median agents reach median: {extra_wins:.0f}")
print(f"Estimated revenue upside: ${upside:,.0f}")
print(f"As % of current won revenue: {upside/won_total:.1%}")

Median agent win rate: 63.6%
Avg value per won deal: $2,361
Extra wins if below-median agents reach median: 101
Estimated revenue upside: $237,433
As % of current won revenue: 2.4%


In [35]:
import os
os.makedirs('../outputs', exist_ok=True)

# full clean master table
df.to_csv('../outputs/master_clean.csv', index=False)

# agent summary for the dashboard
agent_summary = (df[df['is_closed']].groupby(['sales_agent','manager','regional_office'])
                 .agg(deals=('is_won','size'), win_rate=('is_won','mean'),
                      won_rev=('close_value', lambda s: s[df.loc[s.index,'is_won']].sum()))
                 .reset_index())
agent_summary.to_csv('../outputs/agent_summary.csv', index=False)

print("Exported:", os.listdir('../outputs'))

Exported: ['agent_summary.csv', 'master_clean.csv']


In [36]:
lost = df[df['deal_stage']=='Lost']
print(lost.groupby('product')['sales_price'].agg(['count','sum']).sort_values('sum', ascending=False))

                count      sum
product                       
GTX Pro           418  2015178
MG Advanced       430  1458990
GTX Plus Pro      266  1458212
GTX Plus Basic    398   436208
GTX Basic         521   286550
GTK 500            10   267680
MG Special        430    23650
